In [1]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr

!pip install -q google-genai pymupdf pytesseract pillow requests pandas

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 50.8 MB/s eta 0:00:00


In [2]:
import fitz
import pytesseract
import requests
import pandas as pd
import json
import re

from PIL import Image
from google import genai
from google.colab import files
from getpass import getpass

In [5]:
from google import genai
from getpass import getpass

API_KEY = getpass("Enter Gemini API Key: ")

client = genai.Client(api_key=API_KEY)

print("Gemini API configured successfully!")

Enter Gemini API Key: ··········
Gemini API configured successfully!


In [6]:
import fitz
import pytesseract
import requests
import pandas as pd
import json
import re

from PIL import Image
from google import genai
from google.colab import files
from getpass import getpass

In [7]:
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

print("Uploaded file:", file_name)

Saving dummy_medical_lab_report.pdf to dummy_medical_lab_report.pdf
Uploaded file: dummy_medical_lab_report.pdf


In [8]:
def extract_report_text(file_path):

    if file_path.lower().endswith(".pdf"):

        doc = fitz.open(file_path)

        final_text = ""

        for page_number, page in enumerate(doc):

            text = page.get_text("text").strip()

            # Use OCR for scanned PDF
            if len(text) < 50:

                pix = page.get_pixmap(dpi=300)

                image = Image.frombytes(
                    "RGB",
                    [pix.width, pix.height],
                    pix.samples
                )

                text = pytesseract.image_to_string(image)

            final_text += (
                f"\n--- PAGE {page_number + 1} ---\n"
            )

            final_text += text

        return final_text

    else:

        image = Image.open(file_path)

        return pytesseract.image_to_string(image)


report_text = extract_report_text(file_name)

print(report_text[:5000])


--- PAGE 1 ---
SAMPLE MEDICAL LABORATORY REPORT
For Educational and Software Testing Purposes Only
Patient Name
Demo Patient
Age / Sex
24 / Female
Sample ID
TEST-001
Report Type
Complete Blood Count and Basic Lab Panel
Test Name
Result
Unit
Reference Range
Hemoglobin
10.5
g/dL
12.0 - 16.0
White Blood Cell Count
7.2
10^3/uL
4.0 - 11.0
Platelet Count
250
10^3/uL
150 - 450
Fasting Blood Glucose
112
mg/dL
70 - 99
Serum Creatinine
0.9
mg/dL
0.6 - 1.1
TSH
5.8
mIU/L
0.4 - 4.5
Laboratory Note:
Hemoglobin is below the stated reference range. Fasting blood glucose and TSH are above the
stated reference ranges. Laboratory results should be interpreted by a qualified healthcare
professional together with clinical history and other findings.
Disclaimer: This is fictional test data created only for demonstrating a medical report simplification
software project. It is not a real patient report and must not be used for diagnosis or treatment.


In [10]:
extraction_prompt = f"""
You are a medical document information extraction system.

Extract information ONLY from the medical report.

Return valid JSON in this format:

{{
    "report_type": "",
    "medical_terms": [],
    "lab_results": [
        {{
            "test_name": "",
            "value": "",
            "unit": "",
            "reference_low": null,
            "reference_high": null
        }}
    ]
}}

Rules:
1. Do not diagnose.
2. Do not invent values.
3. Use only information available in the report.
4. Use only reference ranges printed in the report.
5. Return JSON only.

MEDICAL REPORT:

{report_text}
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=extraction_prompt
)

raw_json = response.text

raw_json = re.sub(
    r"```json|```",
    "",
    raw_json
).strip()

medical_data = json.loads(raw_json)

print(json.dumps(medical_data, indent=2))

{
  "report_type": "Complete Blood Count and Basic Lab Panel",
  "medical_terms": [
    "Hemoglobin",
    "White Blood Cell Count",
    "Platelet Count",
    "Fasting Blood Glucose",
    "Serum Creatinine",
    "TSH"
  ],
  "lab_results": [
    {
      "test_name": "Hemoglobin",
      "value": "10.5",
      "unit": "g/dL",
      "reference_low": 12.0,
      "reference_high": 16.0
    },
    {
      "test_name": "White Blood Cell Count",
      "value": "7.2",
      "unit": "10^3/uL",
      "reference_low": 4.0,
      "reference_high": 11.0
    },
    {
      "test_name": "Platelet Count",
      "value": "250",
      "unit": "10^3/uL",
      "reference_low": 150,
      "reference_high": 450
    },
    {
      "test_name": "Fasting Blood Glucose",
      "value": "112",
      "unit": "mg/dL",
      "reference_low": 70,
      "reference_high": 99
    },
    {
      "test_name": "Serum Creatinine",
      "value": "0.9",
      "unit": "mg/dL",
      "reference_low": 0.6,
      "reference_high

In [11]:
lab_results = medical_data.get(
    "lab_results",
    []
)

lab_df = pd.DataFrame(lab_results)

lab_df

,test_name,value,unit,reference_low,reference_high
0,Hemoglobin,10.5,g/dL,12.0,16.0
1,White Blood Cell Count,7.2,10^3/uL,4.0,11.0
2,Platelet Count,250,10^3/uL,150.0,450.0
3,Fasting Blood Glucose,112,mg/dL,70.0,99.0
4,Serum Creatinine,0.9,mg/dL,0.6,1.1
5,TSH,5.8,mIU/L,0.4,4.5


In [12]:
def analyze_lab_values(results):

    analyzed_results = []

    for item in results:

        result = item.copy()

        try:

            value = float(item["value"])

            low = item.get("reference_low")
            high = item.get("reference_high")

            if low is not None and value < float(low):

                status = "LOW"

            elif high is not None and value > float(high):

                status = "HIGH"

            elif low is not None and high is not None:

                status = "NORMAL"

            else:

                status = "UNKNOWN"

        except Exception:

            status = "UNKNOWN"

        result["status"] = status

        analyzed_results.append(result)

    return analyzed_results


analyzed_results = analyze_lab_values(
    lab_results
)

analysis_df = pd.DataFrame(
    analyzed_results
)

analysis_df

,test_name,value,unit,reference_low,reference_high,status
0,Hemoglobin,10.5,g/dL,12.0,16.0,LOW
1,White Blood Cell Count,7.2,10^3/uL,4.0,11.0,NORMAL
2,Platelet Count,250,10^3/uL,150.0,450.0,NORMAL
3,Fasting Blood Glucose,112,mg/dL,70.0,99.0,HIGH
4,Serum Creatinine,0.9,mg/dL,0.6,1.1,NORMAL
5,TSH,5.8,mIU/L,0.4,4.5,HIGH


In [13]:
def validate_medical_term(term):

    url = (
        "https://clinicaltables.nlm.nih.gov/"
        "api/conditions/v3/search"
    )

    params = {
        "terms": term,
        "maxList": 3
    }

    try:

        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        if data[0] > 0:

            return data[3]

        return []

    except Exception:

        return []


medical_terms = medical_data.get(
    "medical_terms",
    []
)

validated_terms = {}

for term in medical_terms:

    validated_terms[term] = (
        validate_medical_term(term)
    )


print(
    json.dumps(
        validated_terms,
        indent=2
    )
)

{
  "Hemoglobin": [
    [
      "Stool - heme positive"
    ],
    [
      "Hemoglobin A2 - high"
    ],
    [
      "Hemoglobinopathy"
    ]
  ],
  "White Blood Cell Count": [
    [
      "White blood count - low (leukopenia)"
    ],
    [
      "White blood count - high (leukocytosis)"
    ],
    [
      "Neutropenia"
    ]
  ],
  "Platelet Count": [],
  "Fasting Blood Glucose": [],
  "Serum Creatinine": [],
  "TSH": []
}


In [15]:
simplification_prompt = f"""
You are a medical report simplification assistant.

Explain the following medical report in very simple language.

MEDICAL REPORT:

{report_text}

LAB ANALYSIS:

{json.dumps(analyzed_results, indent=2)}

MEDICAL TERMINOLOGY API RESULTS:

{json.dumps(validated_terms, indent=2)}

Rules:

1. Do not diagnose diseases.
2. Do not prescribe medicines.
3. Do not recommend treatment.
4. Explain difficult medical terms simply.
5. Mention HIGH or LOW only from the lab analysis.
6. Do not exaggerate health risks.
7. Clearly mention unclear information.
8. Encourage discussion with a qualified healthcare professional.

Create the following sections:

REPORT OVERVIEW

LAB RESULT EXPLANATION

MEDICAL TERMS EXPLAINED

IMPORTANT DISCUSSION POINTS
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=simplification_prompt
)

simple_report = response.text

print(simple_report)

### REPORT OVERVIEW

This is a simplified guide to your lab report, which is a **Complete Blood Count and Basic Lab Panel** for a 24-year-old female. 

A lab panel is a routine checkup of different markers in your blood to see how your body is functioning. In this report, most of your results are in the standard healthy range. However, there are three specific areas where the numbers are slightly outside the standard limits: your hemoglobin, your fasting blood glucose, and your TSH (thyroid-stimulating hormone). 

This guide will break down what these terms mean and what these numbers show, so you can easily discuss them with your healthcare provider.

---

### LAB RESULT EXPLANATION

Here is a breakdown of each test from your report, what was measured, and where your results stand:

*   **Hemoglobin: LOW (10.5 g/dL)**
    *   *Standard Range:* 12.0 - 16.0 g/dL
    *   *What this means:* Your level is slightly below the standard range. Hemoglobin is the part of your red blood cells tha

In [17]:
question_prompt = f"""
Based ONLY on the following medical report and lab analysis,
generate 5 useful questions the patient can ask their doctor.

Do not diagnose diseases.
Do not prescribe medicines.
Do not recommend treatment.

MEDICAL REPORT:

{report_text}

LAB ANALYSIS:

{json.dumps(analyzed_results, indent=2)}
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=question_prompt
)

doctor_questions = response.text

print(doctor_questions)

Based on your lab report and analysis, here are 5 questions you can ask your doctor:

1. **Regarding my low hemoglobin:** "My hemoglobin level is low at 10.5 g/dL. What are the potential causes for this below-range result, and are there additional follow-up tests you recommend to investigate it?"
2. **Regarding my high fasting blood glucose:** "My fasting blood glucose is elevated at 112 mg/dL. What does this high level indicate, and what steps or additional testing (such as an HbA1c test) do you suggest to monitor this?"
3. **Regarding my high TSH:** "My TSH level is high at 5.8 mIU/L. What does this result mean for my thyroid function, and should we run any further thyroid panels?"
4. **Regarding the overall pattern of my results:** "Since my white blood cell count (7.2), platelet count (250), and serum creatinine (0.9) are all normal, but my hemoglobin, glucose, and TSH are out of range, do these abnormal results relate to one another in any way?"
5. **Regarding next steps:** "Based

In [18]:
print("=" * 60)

print("AI MEDICAL REPORT SIMPLIFIER")

print("=" * 60)

print("\nSIMPLE REPORT EXPLANATION\n")

print(simple_report)

print("\n" + "=" * 60)

print("QUESTIONS TO ASK YOUR DOCTOR\n")

print(doctor_questions)

print("\n" + "=" * 60)

print(
    "DISCLAIMER: This application simplifies medical "
    "information and does not provide medical diagnosis "
    "or treatment."
)

AI MEDICAL REPORT SIMPLIFIER

SIMPLE REPORT EXPLANATION

### REPORT OVERVIEW

This is a simplified guide to your lab report, which is a **Complete Blood Count and Basic Lab Panel** for a 24-year-old female. 

A lab panel is a routine checkup of different markers in your blood to see how your body is functioning. In this report, most of your results are in the standard healthy range. However, there are three specific areas where the numbers are slightly outside the standard limits: your hemoglobin, your fasting blood glucose, and your TSH (thyroid-stimulating hormone). 

This guide will break down what these terms mean and what these numbers show, so you can easily discuss them with your healthcare provider.

---

### LAB RESULT EXPLANATION

Here is a breakdown of each test from your report, what was measured, and where your results stand:

*   **Hemoglobin: LOW (10.5 g/dL)**
    *   *Standard Range:* 12.0 - 16.0 g/dL
    *   *What this means:* Your level is slightly below the standard 